In [5]:
import warnings
warnings.filterwarnings('ignore')

import DestinyResearch as dr
from DestinyResearch.features.vpin import get_vpin
from DestinyResearch.features.vpin import get_vpin_range

In [2]:
df = dr.get_mbo_front_rth("ES", "2025-10-10")

[dr] Front month for ES on 2025-10-10: ESZ25


In [4]:
df.to_pandas().head()

,ts_event,ts_recv,venue,product,contract,action,side,price,size,order_id,flags,sequence,publisher_id,instrument_id
0,1760102999999979383,1760103000000078194,CME,ES,ESZ25,ADD,ASK,6788250000000,1,6415635508623,128,38257008,1,294973
1,1760102999999994819,1760103000000091533,CME,ES,ESZ25,ADD,ASK,6788250000000,4,6415635508624,128,38257009,1,294973
2,1760103000000128657,1760103000000907743,CME,ES,ESZ25,ADD,ASK,6788250000000,1,6415635508625,128,38257010,1,294973
3,1760103000000174311,1760103000001054217,CME,ES,ESZ25,TRADE,ASK,6788000000000,2,6415635508626,0,38257011,1,294973
4,1760103000000174311,1760103000001054217,CME,ES,ESZ25,FILL,BID,6788000000000,1,6415635506641,0,38257011,1,294973


# Test single day

In [2]:
vpin_df = get_vpin("ES", "2025-10-10", bucket_pct=0.05, window=50)

[dr] Front month for ES on 2025-10-10: ESZ25


In [3]:
print(f"Total buckets: {len(vpin_df)}")
print(f"\nVPIN stats:")
print(vpin_df['vpin'].describe())

print(f"\nFirst 5 buckets (warmup):")
print(vpin_df[['bucket_id', 'vpin', 'order_imbalance', 'v_buy', 'v_sell']].head())

print(f"\nLast 5 buckets (full window):")
print(vpin_df[['bucket_id', 'vpin', 'order_imbalance', 'v_buy', 'v_sell']].tail())

# Check buy/sell balance
total_buy = vpin_df['v_buy'].sum()
total_sell = vpin_df['v_sell'].sum()
print(f"\nTotal volume balance:")
print(f"  BUY:  {total_buy:,} contracts ({total_buy/(total_buy+total_sell)*100:.1f}%)")
print(f"  SELL: {total_sell:,} contracts ({total_sell/(total_buy+total_sell)*100:.1f}%)")

Total buckets: 21

VPIN stats:
count    21.000000
mean      0.029236
std       0.011116
min       0.011254
25%       0.026201
50%       0.028296
75%       0.031024
max       0.071808
Name: vpin, dtype: float64

First 5 buckets (warmup):
   bucket_id      vpin  order_imbalance  v_buy  v_sell
0          0  0.011254         0.011254  53683   54905
1          1  0.016309         0.021365  55455   53135
2          2  0.021992         0.033358  52493   56116
3          3  0.032324         0.063318  57735   50859
4          4  0.028296         0.012183  53637   54960

Last 5 buckets (full window):
    bucket_id      vpin  order_imbalance  v_buy  v_sell
16         16  0.025535         0.002173  54416   54180
17         17  0.026201         0.037525  52260   56335
18         18  0.026346         0.028947  55861   52718
19         19  0.025399         0.007405  53882   54686
20         20  0.071808         1.000000     61       0

Total volume balance:
  BUY:  1,089,668 contracts (50.2%)
  SELL:

# Test multi-day continuous VPIN

In [6]:
vpin_range = get_vpin_range("ES", "2025-10-01", "2025-10-10", bucket_pct=0.05, window=50)

[dr] Front month for ES on 2025-10-01: ESZ25
[dr] Front month for ES on 2025-10-02: ESZ25
[dr] Front month for ES on 2025-10-03: ESZ25
[dr] Front month for ES on 2025-10-05: ESZ25
[dr] Front month for ES on 2025-10-06: ESZ25
[dr] Front month for ES on 2025-10-07: ESZ25
[dr] Front month for ES on 2025-10-08: ESZ25
[dr] Front month for ES on 2025-10-09: ESZ25
[dr] Front month for ES on 2025-10-10: ESZ25


In [7]:
print(f"Total buckets: {len(vpin_range)}")
print(f"Days: {vpin_range['date'].unique()}")
print(f"\nWarmup buckets: {vpin_range['is_warmup'].sum()}")
print(f"Full-window buckets: {(~vpin_range['is_warmup']).sum()}")

print(f"\nVPIN stats (full window only):")
print(vpin_range[~vpin_range['is_warmup']]['vpin'].describe())

print(f"\nFirst day transition:")
print(vpin_range[vpin_range['date'] == '2025-10-08'].tail(3))
print(vpin_range[vpin_range['date'] == '2025-10-09'].head(3))

print(f"\nBucket ID continuity:")
print(f"  Day 1 last bucket: {vpin_range[vpin_range['date'] == '2025-10-08']['bucket_id_global'].max()}")
print(f"  Day 2 first bucket: {vpin_range[vpin_range['date'] == '2025-10-09']['bucket_id_global'].min()}")

Total buckets: 168
Days: <ArrowStringArray>
['2025-10-01', '2025-10-02', '2025-10-03', '2025-10-06', '2025-10-07',
 '2025-10-08', '2025-10-09', '2025-10-10']
Length: 8, dtype: str

Warmup buckets: 49
Full-window buckets: 119

VPIN stats (full window only):
count    119.000000
mean       0.057755
std        0.014085
min        0.029499
25%        0.046415
50%        0.056682
75%        0.066586
max        0.086667
Name: vpin, dtype: float64

First day transition:
           date  bucket_id_global  bucket_id_daily              ts_last  \
123  2025-10-08               123               18  1759953539067676127   
124  2025-10-08               124               19  1759953599996518341   
125  2025-10-08               125               20  1759953599999516135   

     v_buy  v_sell  v_total  n_trades  order_imbalance      vpin  is_warmup  
123  19636   21198    40834      7499         0.038252  0.052832      False  
124  18867   21971    40838      5365         0.076008  0.053778      False 